# Managing Big Data

In [1]:
# connect to Enterprise GIS
from arcgis.gis import GIS
import arcgis.geoanalytics

portal_gis = GIS("https://ndhwks6.esri.com/portal", "admin", 'esri.agp', verify_cert=False)

In [2]:
item = portal_gis.content.get('c81431e0194a42cc9e6f4336c76a70b6')
usa_counties_lyr = item.layers[0]

In [3]:
usa_counties_lyr

<FeatureLayer url:"https://ndhwks6.esri.com/server/rest/services/Hosted/usaCounties/FeatureServer/0">

In [4]:
bigdata_datastore_manager = arcgis.geoanalytics.get_datastores()
bigdata_datastore_manager

<DatastoreManager for https://ndhwks6.esri.com:6443/arcgis/admin>

In [67]:
data_item2 = bigdata_datastore_manager.add_bigdata("air_quality", r"\\NDHWKS6\Users\arcgis\Documents\air")

Big Data file share exists for air_quality


In [5]:
search_result1 = portal_gis.content.search("bigDataFileShares_air_quality", item_type = "big data file share")
search_result1

[<Item title:"bigDataFileShares_air_quality" type:Big Data File Share owner:admin>]

In [6]:
air_lyr = search_result1[0].layers[0]

In [7]:
air_lyr

<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_air_quality/BigDataCatalogServer/air_quality">

In [8]:
search_result2 = portal_gis.content.search("bigDataFileShares_all_hurricanes", item_type = "big data file share")
search_result2

[<Item title:"bigDataFileShares_all_hurricanes" type:Big Data File Share owner:admin>]

In [9]:
hurr = search_result2[0].layers[0]

In [10]:
search_result3 = portal_gis.content.search("bigDataFileShares_ServiceCallsOrleans", item_type = "big data file share")[0]
search_result3

<Item title:"bigDataFileShares_ServiceCallsOrleans" type:Big Data File Share owner:admin>

In [11]:
calls = search_result3.layers[0]

In [13]:
from arcgis.features import FeatureLayer

In [55]:
blk_grp_lyr = FeatureLayer('https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/USA_Census_BlockGroup_Areas_analysis_trim/FeatureServer/0')

In [56]:
blk_grp_lyr.filter = "County='Orleans'"

## Append Data

In [13]:
from arcgis.geoanalytics.manage_data import append_data

input_lyr = portal_gis.content.get('b39deed705144a2a90c5eaf5a44f5a14').layers[0]
append_lyr = portal_gis.content.get('78d4b1914bba4e09b9e8006fa6a3157c').layers[0]

append_data(input_layer=input_lyr, append_layer=append_lyr)

## Calculate Fields

In [14]:
from arcgis.geoanalytics.manage_data import calculate_fields

In [19]:
calculate_fields(input_layer=hurr,
                 field_name="avg",
                 data_type="Double",
                 expression='max($feature["wind_wmo1"],$feature["pres_wmo1"])')

{"messageCode":"BD_101051","message":"Possible issues were found while reading 'inputLayer'.","params":{"paramName":"inputLayer"}}
{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}


<Item title:"Calculate_Field_4BF9US" type:Feature Layer Collection owner:admin>

## Clip Layer

In [16]:
from arcgis.geoanalytics.manage_data import clip_layer
from datetime import datetime as dt

In [17]:
clip_result = clip_layer(calls, blk_grp_lyr, output_name="service calls in new Orleans" + str(dt.now().microsecond))

{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}
{"messageCode":"BD_101054","message":"Some records have either missing or invalid geometries."}


## Copy To Datastore

In [18]:
from arcgis.geoanalytics.manage_data import copy_to_data_store

In [19]:
copy_to_data_store(hurr)

{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}


<Item title:"Copy_to_Data_Store_9493ZL" type:Feature Layer Collection owner:admin>

## Dissolve Boundaries

In [51]:
from arcgis.geoanalytics.manage_data import dissolve_boundaries

In [58]:
dissolve_boundaries(input_layer=blk_grp_lyr, 
                    dissolve_fields='County', 
                    output_name='dissolved by countyfp')

<Item title:"dissolved_by_countyfp" type:Feature Layer Collection owner:admin>

## Merge Layers

In [76]:
from arcgis.geoanalytics.manage_data import merge_layers

In [63]:
bigdata_datastore_manager.add_bigdata("nyc1", r"\\NDHWKS6\Users\arcgis\Documents\NYC_taxi_data1")
bigdata_datastore_manager.add_bigdata("nyc2", r"\\NDHWKS6\Users\arcgis\Documents\NYC_taxi_data2")

Created Big Data file share for nyc1
Created Big Data file share for nyc2


<Datastore title:"/bigDataFileShares/nyc2" type:"bigDataFileShare">

In [73]:
taxi_search = portal_gis.content.search("bigDataFileShares_nyc", item_type = "big data file share")
taxi_search

[<Item title:"bigDataFileShares_nyc1" type:Big Data File Share owner:admin>,
 <Item title:"bigDataFileShares_nyc2" type:Big Data File Share owner:admin>]

In [74]:
taxi1 = taxi_search[0].layers[0]
taxi2 = taxi_search[1].layers[0]

In [ ]:
merge_layers(taxi1, taxi2, output_name='merged layers'+ str(dt.now().microsecond))

## Overlay Data

In [ ]:
from arcgis.geoanalytics.manage_data import overlay_data

In [ ]:
overlay_data(calls, blk_grp_lyr, output_name='intersected features'+ str(dt.now().microsecond))

## Run Python Script

In [24]:
from arcgis.geoanalytics.manage_data import run_python_script

The function below filters the data by rows that give information about PM2.5 pollutant. To find the average PM2.5 value of each county, we will use `join_features` tool. Finally, we will write the output to the datastore.

In [25]:
def average():
    spark_df = layers[0] #
    spark_df = spark_df.filter(spark_df['Parameter Name'] == 'PM2.5 - Local Conditions') #pyspark filter
    res = geoanalytics.join_features(target_layer=layers[1],  
                                     join_layer=spark_df, 
                                     join_operation="JoinOneToOne",
                                     summary_fields=[{'statisticType' : 'mean', 'onStatisticField' : 'Sample Measurement'}],
                                     spatial_relationship='Contains')
    res.write.format("webgis").save("average_pm_by_county")

In [26]:
run_python_script(average, [air_lyr, usa_counties_lyr])

[{'type': 'esriJobMessageTypeInformative',
  'description': 'Start Time: Thursday, October 28, 2021 2:20:18 PM'},
 {'type': 'esriJobMessageTypeInformative',
  'description': 'Using URL based GPRecordSet param: https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_air_quality/BigDataCatalogServer/air_quality'},
 {'type': 'esriJobMessageTypeInformative',
  'description': 'Using URL based GPRecordSet param: https://ndhwks6.esri.com/server/rest/services/Hosted/usaCounties/FeatureServer/0'},
 {'type': 'esriJobMessageTypeInformative',
  'description': '{"messageCode":"BD_101028","message":"Starting new distributed job with 3441 tasks.","params":{"totalTasks":"3441"}}'},
 {'type': 'esriJobMessageTypeInformative',
  'description': '{"messageCode":"BD_101029","message":"0/3441 distributed tasks completed.","params":{"completedTasks":"0","totalTasks":"3441"}}'},
 {'type': 'esriJobMessageTypeInformative',
  'description': '{"messageCode":"BD_101029","message":"1/3441 d